# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinivas25046/FlyRank-MLstarter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring**

My lane is **Refresh / Content Opportunity Scoring**, and the underlying decision from ML-02 is *"which pages should be reviewed first?"* — not *"is this page declining, yes or no?"*.
The `framing-ml-problems` table maps "which ones first?" straight to **ranking/scoring**, with a priority score as the target, not a class label.

Classification is tempting because `trend_direction == "down"` already looks like a ready-made label. But a binary label throws away magnitude: a page that dropped 90% while pulling 50,000 impressions and a page that dropped 22% on 20 impressions both get labeled `1`, even though no reviewer would treat them the same. The manager needs an ordered queue, not a yes/no verdict — that's what makes this scoring/ranking rather than classification.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

RAW_URL = (
    "https://raw.githubusercontent.com/Srinivas25046/FlyRank-MLstarter/"
    "refs/heads/main/data/raw/content_refresh_anonymized.csv"
)
LOCAL_PATH = "../../data/raw/content_refresh_anonymized.csv"

try:
    # Works when running from a real clone of the repo (e.g. work/notebooks/ locally)
    df = pd.read_csv(LOCAL_PATH)
except FileNotFoundError:
    # Works in Colab, or anywhere else the repo isn't checked out on disk
    df = pd.read_csv(RAW_URL)

# Show why a binary label alone can't support "which ones first?"
declining = df[df["trend_direction"] == "down"]
print(f"Pages labeled 'down': {len(declining):,} ({len(declining)/len(df)*100:.1f}% of all pages)")
print("\nAll of these get the SAME binary label, but look at the spread in traffic at stake:")
print(declining["impressions_90d"].describe()[["min", "25%", "50%", "75%", "max"]])
print(
    "\n-> One classification bucket spans from 1 impression to "
    f"{int(declining['impressions_90d'].max()):,} impressions in 90 days. "
    "A single label can't order these — a score can."
)

Pages labeled 'down': 16,262 (54.2% of all pages)

All of these get the SAME binary label, but look at the spread in traffic at stake:
min         1.00
25%       179.00
50%       961.00
75%      3831.75
max    517715.00
Name: impressions_90d, dtype: float64

-> One classification bucket spans from 1 impression to 517,715 impressions in 90 days. A single label can't order these — a score can.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `priority_score` (a continuous proxy, not an observed outcome)**

No column in this dataset records "a human reviewed this page and it was worth it" — that
outcome has never been observed here, so I can't pretend to have a real target yet. What I can
build is an honest **proxy**, following the two framing rules (name the metric, and don't let a
defined rule pretend to be the world):

- **Decline signal** — same definition the data dictionary gives for `is_declining_label`
  (`trend_direction == "down"`). The raw CSV doesn't ship that derived column, so I rebuild it
  here from `trend_direction` directly, in the open.
- **Traffic at stake** — `impressions_90d` and `clicks_90d`, because a declining page nobody
  searches for isn't worth a reviewer's hour.
- **Measurability gate** — mirrors `measurable_opportunity` in the dictionary
  (`impressions_90d >= 100` and `sessions_90d > 0`), so tiny-sample noise doesn't rank a page
  highly by accident.

`priority_score = is_declining * log1p(impressions_90d + clicks_90d) * measurable_opportunity`

This is a **defined proxy**, not an observed one — I'm saying so explicitly rather than dressing
it up. The honest path to an observed target: once the FlyRank warehouse's
`fact_content_daily_performance` table is in play (week 4+), I could check whether pages that
were actually reviewed/refreshed saw impressions recover in the *following* 30–90 day window,
using only `*_prev30`-style columns to avoid leakage. That's next week's job, not this one's.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.